# 01 — MODFLOW 6 Baseline Run Check

**Purpose:** Run the copied Little Plover River MODFLOW 6 steady-state model from this repository.

This notebook checks that the model can be:

1. Found in the clean source model folder.
2. Copied into a safe `results/` run folder.
3. Loaded with FloPy.
4. Run with the `mf6` executable.
5. Checked for successful completion.
6. Summarized using the generated output files.

## Important idea

We will **not** run the model directly inside:

```text
models/lpr_mf6/steady_state/
```

That folder is treated as the clean, original source model copied from `LPR_redux`.

Instead, this notebook copies the model into:

```text
results/mf6_runs/baseline_steady_state/
```

and runs MODFLOW there. This keeps the original model files clean.


## 1. Imports and project-root setup

In [ ]:
from pathlib import Path
import os
import sys
import shutil
import subprocess
from datetime import datetime
import pandas as pd

def find_project_root(start=None):
    """Find the repository root by looking for common project marker files."""
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start] + list(start.parents)

    for candidate in candidates:
        if (candidate / ".git").exists() and (candidate / "environment.yml").exists():
            return candidate

    for candidate in candidates:
        if (candidate / "models").exists() and (candidate / "data").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. Try opening Jupyter from the Modeling-Uncertainties repo root."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Current working directory: {Path.cwd()}")
print(f"Python executable: {sys.executable}")


## 2. Define source and run paths

In [ ]:
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
MF6_RESULTS_DIR = RESULTS_DIR / "mf6_runs"

# Clean source model copied from LPR_redux
LPR_MF6_SOURCE_DIR = MODELS_DIR / "lpr_mf6" / "steady_state"

# Safe run workspace for this notebook
BASELINE_RUN_DIR = MF6_RESULTS_DIR / "baseline_steady_state"

print(f"Source model folder: {LPR_MF6_SOURCE_DIR}")
print(f"Baseline run folder: {BASELINE_RUN_DIR}")


## 3. Check the source MODFLOW 6 model files

This checks the clean source folder before copying it into the run folder.


In [ ]:
required_mf6_files = [
    "mfsim.nam",
    "lpr_ss.tdis",
    "lpr_ss.ims",
    "lpr_ss_gwf.nam",
    "lpr_ss_gwf.dis",
    "lpr_ss_gwf.npf",
    "lpr_ss_gwf.ic",
    "lpr_ss_gwf.oc",
    "lpr_ss_gwf.rcha",
    "lpr_ss_gwf.wel",
    "lpr_ss_gwf.drn",
    "lpr_ss_gwf.chd",
    "lpr_ss.sfr",
    "sfr_obs.obs",
]

source_file_check = pd.DataFrame(
    [
        {
            "file": filename,
            "relative_path": str((LPR_MF6_SOURCE_DIR / filename).relative_to(PROJECT_ROOT)),
            "exists": (LPR_MF6_SOURCE_DIR / filename).exists(),
            "size_MB": round((LPR_MF6_SOURCE_DIR / filename).stat().st_size / 1_000_000, 3)
            if (LPR_MF6_SOURCE_DIR / filename).exists()
            else None,
        }
        for filename in required_mf6_files
    ]
)

source_file_check


In [ ]:
missing_files = source_file_check.loc[~source_file_check["exists"], "file"].tolist()

if missing_files:
    raise FileNotFoundError(f"Missing required MODFLOW 6 source files: {missing_files}")

print("All required MODFLOW 6 source files were found.")
print(f"Total files in source model folder: {len(list(LPR_MF6_SOURCE_DIR.glob('*')))}")


## 4. Copy the source model into a safe run folder

Set `OVERWRITE_RUN_FOLDER = True` to make this notebook repeatable.

This will delete and recreate only:

```text
results/mf6_runs/baseline_steady_state/
```

It will **not** modify the clean source folder.


In [ ]:
OVERWRITE_RUN_FOLDER = True

if BASELINE_RUN_DIR.exists():
    if OVERWRITE_RUN_FOLDER:
        print(f"Removing existing run folder: {BASELINE_RUN_DIR.relative_to(PROJECT_ROOT)}")
        shutil.rmtree(BASELINE_RUN_DIR)
    else:
        raise FileExistsError(
            f"The run folder already exists: {BASELINE_RUN_DIR}. "
            "Set OVERWRITE_RUN_FOLDER = True if you want to replace it."
        )

BASELINE_RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LPR_MF6_SOURCE_DIR, BASELINE_RUN_DIR)

print(f"Copied source model to: {BASELINE_RUN_DIR.relative_to(PROJECT_ROOT)}")
print(f"Total files in run folder: {len(list(BASELINE_RUN_DIR.glob('*')))}")


## 5. Inspect the copied run folder

In [ ]:
run_files = sorted([p for p in BASELINE_RUN_DIR.iterdir() if p.is_file()])

run_file_table = pd.DataFrame(
    [
        {
            "file": p.name,
            "size_MB": round(p.stat().st_size / 1_000_000, 3),
        }
        for p in run_files
    ]
)

run_file_table.head(20)


## 6. Load the copied model with FloPy

This does not run MODFLOW yet. It only checks that FloPy can read the copied input files.


In [ ]:
import flopy

sim = flopy.mf6.MFSimulation.load(
    sim_ws=str(BASELINE_RUN_DIR),
    verbosity_level=1,
)

print("MODFLOW 6 simulation loaded successfully with FloPy.")
print(f"Simulation name: {sim.name}")
print(f"Model names: {sim.model_names}")

gwf = sim.get_model()
print(f"Groundwater model name: {gwf.name}")

print("\nPackages:")
for package in gwf.packagelist:
    print(f"  - {package.package_type:8s} {package.filename}")


## 7. Basic model information from FloPy

In [ ]:
# Discretization information
dis = gwf.get_package("dis")

nlay = int(dis.nlay.get_data())
nrow = int(dis.nrow.get_data())
ncol = int(dis.ncol.get_data())

print(f"Number of layers: {nlay}")
print(f"Number of rows:   {nrow}")
print(f"Number of cols:   {ncol}")
print(f"Total cells:      {nlay * nrow * ncol:,}")

# Stress period information from TDIS
tdis = sim.get_package("tdis")
perioddata = tdis.perioddata.get_data()

period_table = pd.DataFrame(
    perioddata,
    columns=["perlen", "nstp", "tsmult"]
)

period_table


## 8. Check the `mf6` executable

Before running the model, make sure the `mf6` executable is available from the active environment.


In [ ]:
mf6_path = shutil.which("mf6")

if mf6_path is None:
    raise FileNotFoundError(
        "The mf6 executable was not found. Activate your project environment with: conda activate gw_uncertainty"
    )

print(f"mf6 executable found at: {mf6_path}")

version_result = subprocess.run(
    ["mf6", "-v"],
    capture_output=True,
    text=True,
    timeout=30,
)

print("mf6 version output:")
print(version_result.stdout.strip() or version_result.stderr.strip())


## 9. Run MODFLOW 6 in the copied run folder

This is the first real model run from the clean repository workflow.

The model will run inside:

```text
results/mf6_runs/baseline_steady_state/
```


In [ ]:
start_time = datetime.now()

run_result = subprocess.run(
    ["mf6"],
    cwd=str(BASELINE_RUN_DIR),
    capture_output=True,
    text=True,
    timeout=300,
)

end_time = datetime.now()
elapsed = end_time - start_time

stdout_path = BASELINE_RUN_DIR / "run_stdout.txt"
stderr_path = BASELINE_RUN_DIR / "run_stderr.txt"

stdout_path.write_text(run_result.stdout, encoding="utf-8")
stderr_path.write_text(run_result.stderr, encoding="utf-8")

print(f"Return code: {run_result.returncode}")
print(f"Elapsed time: {elapsed}")
print(f"Saved stdout to: {stdout_path.relative_to(PROJECT_ROOT)}")
print(f"Saved stderr to: {stderr_path.relative_to(PROJECT_ROOT)}")

print("\n--- Last 40 lines of MODFLOW stdout ---")
stdout_lines = run_result.stdout.splitlines()
for line in stdout_lines[-40:]:
    print(line)

if run_result.stderr.strip():
    print("\n--- MODFLOW stderr ---")
    print(run_result.stderr)


## 10. Check whether the MODFLOW run succeeded

In [ ]:
stdout_lower = run_result.stdout.lower()
stderr_lower = run_result.stderr.lower()

normal_termination = "normal termination" in stdout_lower
return_code_ok = run_result.returncode == 0

run_success = return_code_ok and normal_termination

run_summary = pd.DataFrame(
    [
        {"check": "Return code is 0", "passed": return_code_ok},
        {"check": "MODFLOW reported normal termination", "passed": normal_termination},
        {"check": "Overall run success", "passed": run_success},
    ]
)

run_summary


In [ ]:
if not run_success:
    raise RuntimeError(
        "MODFLOW 6 did not clearly report a successful run. "
        "Check run_stdout.txt and run_stderr.txt in the baseline run folder."
    )

print("MODFLOW 6 baseline run completed successfully.")


## 11. List generated output files

This compares files before and after the run, then lists likely output files.


In [ ]:
all_run_files = sorted([p for p in BASELINE_RUN_DIR.iterdir() if p.is_file()])

output_file_table = pd.DataFrame(
    [
        {
            "file": p.name,
            "suffix": p.suffix,
            "size_MB": round(p.stat().st_size / 1_000_000, 3),
            "modified_time": datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec="seconds"),
        }
        for p in all_run_files
    ]
).sort_values(["suffix", "file"])

output_file_table


In [ ]:
likely_output_suffixes = {
    ".hds",
    ".hed",
    ".cbc",
    ".bud",
    ".lst",
    ".csv",
    ".out",
}

likely_outputs = output_file_table[
    output_file_table["suffix"].str.lower().isin(likely_output_suffixes)
    | output_file_table["file"].str.lower().str.contains("budget|head|list|obs|stdout|stderr")
].copy()

likely_outputs


## 12. Inspect the listing file, if available

The listing file is often one of the most useful first checks after a MODFLOW run.


In [ ]:
listing_candidates = sorted(BASELINE_RUN_DIR.glob("*.lst"))

if not listing_candidates:
    print("No .lst listing file found in the run folder.")
else:
    listing_path = listing_candidates[0]
    print(f"Listing file found: {listing_path.relative_to(PROJECT_ROOT)}")
    print(f"Size: {listing_path.stat().st_size / 1_000_000:.3f} MB")

    listing_text = listing_path.read_text(errors="ignore")
    listing_lines = listing_text.splitlines()

    print("\n--- Last 60 lines of listing file ---")
    for line in listing_lines[-60:]:
        print(line)


## 13. Try reading head output with FloPy

Depending on the output-control settings, a binary head file may or may not be produced.

This cell looks for common head-file extensions and reads the first matching file.


In [ ]:
from flopy.utils import HeadFile

head_candidates = sorted(list(BASELINE_RUN_DIR.glob("*.hds")) + list(BASELINE_RUN_DIR.glob("*.hed")))

if not head_candidates:
    print("No binary head file found. This may be okay depending on the OC package settings.")
else:
    head_path = head_candidates[0]
    print(f"Head file found: {head_path.relative_to(PROJECT_ROOT)}")

    hds = HeadFile(str(head_path))
    times = hds.get_times()
    print(f"Number of output times: {len(times)}")
    print(f"Output times: {times}")

    head = hds.get_data(totim=times[-1])
    print(f"Head array shape: {head.shape}")
    print(f"Minimum head: {head.min():.3f}")
    print(f"Maximum head: {head.max():.3f}")


## 14. Try reading budget output with FloPy

This checks for a binary cell-budget file.


In [ ]:
from flopy.utils import CellBudgetFile

budget_candidates = sorted(list(BASELINE_RUN_DIR.glob("*.cbc")) + list(BASELINE_RUN_DIR.glob("*.bud")))

if not budget_candidates:
    print("No binary budget file found. This may be okay depending on the OC package settings.")
else:
    budget_path = budget_candidates[0]
    print(f"Budget file found: {budget_path.relative_to(PROJECT_ROOT)}")

    cbb = CellBudgetFile(str(budget_path), precision="double")
    records = cbb.get_unique_record_names()
    times = cbb.get_times()

    print(f"Number of budget output times: {len(times)}")
    print("Budget record names:")
    for record in records:
        print(f"  - {record}")


## 15. Save a baseline run summary table

The summary is saved to:

```text
results/mf6_runs/baseline_steady_state_run_summary.csv
```


In [ ]:
summary_rows = [
    {"item": "project_root", "value": str(PROJECT_ROOT)},
    {"item": "source_model_folder", "value": str(LPR_MF6_SOURCE_DIR.relative_to(PROJECT_ROOT))},
    {"item": "baseline_run_folder", "value": str(BASELINE_RUN_DIR.relative_to(PROJECT_ROOT))},
    {"item": "python_executable", "value": sys.executable},
    {"item": "mf6_executable", "value": mf6_path},
    {"item": "mf6_return_code", "value": run_result.returncode},
    {"item": "normal_termination", "value": normal_termination},
    {"item": "run_success", "value": run_success},
    {"item": "elapsed_time", "value": str(elapsed)},
    {"item": "nlay", "value": nlay},
    {"item": "nrow", "value": nrow},
    {"item": "ncol", "value": ncol},
    {"item": "total_cells", "value": nlay * nrow * ncol},
    {"item": "run_timestamp", "value": end_time.isoformat(timespec="seconds")},
]

baseline_summary = pd.DataFrame(summary_rows)
summary_path = MF6_RESULTS_DIR / "baseline_steady_state_run_summary.csv"
baseline_summary.to_csv(summary_path, index=False)

print(f"Saved summary to: {summary_path.relative_to(PROJECT_ROOT)}")
baseline_summary


## 16. Final interpretation

If this notebook ran successfully, then:

```text
models/lpr_mf6/steady_state/
```

is a valid clean source model folder, and:

```text
results/mf6_runs/baseline_steady_state/
```

is a valid runnable MODFLOW 6 workspace.

Recommended next steps:

1. Commit the executed notebook if you want GitHub to show the successful run.
2. Do **not** commit large generated binary outputs unless you intentionally want them version-controlled.
3. Start a PyCAP baseline run/check notebook next:

```text
02_pycap_baseline_run_check.ipynb
```

That next notebook will confirm that the analytical model side of the project can be rerun or inspected cleanly.
